# Fine-tuning: st-polish-paraphrase-from-mpnet → st-polish-przemkow

Notatnik trenuje model `sdadas/st-polish-paraphrase-from-mpnet` na parach pytanie–fragment z archiwum Przemkowa.

**Dane wejściowe** (wgraj ręcznie na Dysk Google lub przez `files.upload()`):
- `pary_claude.jsonl` — 5 758 par (Claude Haiku)
- `pary_qwen_czyste.jsonl` — 5 741 par (Qwen 2.5:14b, przefiltrowane)

**Architektura treningu**: MultipleNegativesRankingLoss (MNR)  
Każda para `(pytanie, tekst)` traktowana jest jako pozytywna.  
Pozostałe przykłady w batchu służą jako hard negatives — bez potrzeby ręcznego oznaczania.

**Środowisko**: Google Colab T4 (bezpłatny) lub A100 (Colab Pro)

## 1. Instalacja zależności

In [ ]:
%%capture
!pip install sentence-transformers==3.3.1 datasets accelerate

## 2. Montowanie Dysku Google i wgrywanie plików

**Opcja A** – pliki już na Dysku Google (zalecane):

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Ustaw ścieżkę do folderu z plikami na Dysku
DRIVE_DIR = '/content/drive/MyDrive/historyk_ai'  # <- zmień jeśli trzeba

import os
PARY_CLAUDE = os.path.join(DRIVE_DIR, 'pary_claude.jsonl')
PARY_QWEN   = os.path.join(DRIVE_DIR, 'pary_qwen_czyste.jsonl')

print('Pliki dostępne:', os.path.exists(PARY_CLAUDE), os.path.exists(PARY_QWEN))

In [ ]:
# Opcja B – wgranie ręczne (zakomentuj jeśli używasz Opcji A)
# from google.colab import files
# uploaded = files.upload()  # wybierz oba pliki
# PARY_CLAUDE = 'pary_claude.jsonl'
# PARY_QWEN   = 'pary_qwen_czyste.jsonl'

## 3. Wczytanie i scalenie danych

In [ ]:
import json
import random

def load_pairs(path):
    pairs = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            pytanie = obj.get('pytanie', '').strip()
            tekst   = obj.get('tekst', '').strip()
            if pytanie and tekst:
                pairs.append((pytanie, tekst))
    return pairs

pairs_claude = load_pairs(PARY_CLAUDE)
pairs_qwen   = load_pairs(PARY_QWEN)

all_pairs = pairs_claude + pairs_qwen

# Deduplikacja po kluczu (pytanie, pierwsze 80 znaków tekstu)
seen = set()
deduped = []
for q, t in all_pairs:
    key = (q.lower(), t[:80].lower())
    if key not in seen:
        seen.add(key)
        deduped.append((q, t))

random.seed(42)
random.shuffle(deduped)

# Podział 95/5
split = int(len(deduped) * 0.95)
train_pairs = deduped[:split]
eval_pairs  = deduped[split:]

print(f'Claude:  {len(pairs_claude):>6} par')
print(f'Qwen:    {len(pairs_qwen):>6} par')
print(f'Łącznie: {len(deduped):>6} par (po deduplikacji)')
print(f'Train:   {len(train_pairs):>6}')
print(f'Eval:    {len(eval_pairs):>6}')

## 4. Przygotowanie datasetu dla sentence-transformers

In [ ]:
from datasets import Dataset
from sentence_transformers import SentenceTransformer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.evaluation import InformationRetrievalEvaluator

# Dataset w formacie (anchor, positive) dla MNR Loss
train_ds = Dataset.from_dict({
    'anchor':   [q for q, t in train_pairs],
    'positive': [t for q, t in eval_pairs[:1] * 0 + train_pairs],  # workaround
})
# Poprawna wersja:
train_ds = Dataset.from_dict({
    'anchor':   [q for q, t in train_pairs],
    'positive': [t for q, t in train_pairs],
})
eval_ds = Dataset.from_dict({
    'anchor':   [q for q, t in eval_pairs],
    'positive': [t for q, t in eval_pairs],
})

print('Train dataset:', train_ds)
print('Eval dataset:', eval_ds)

## 5. Wczytanie modelu bazowego

In [ ]:
MODEL_ID = 'sdadas/st-polish-paraphrase-from-mpnet'

model = SentenceTransformer(MODEL_ID)
print(f'Model załadowany. Wymiar embeddingu: {model.get_sentence_embedding_dimension()}')
print(f'Max sekwencja: {model.max_seq_length} tokenów')

## 6. Ewaluator Information Retrieval

Mierzy Recall@1, Recall@5 i MRR@10 — czyli czy właściwy fragment trafia w top-K dla danego pytania.

In [ ]:
# Budujemy korpus z fragmentów ewaluacyjnych
# Klucze muszą być stringami
corpus = {
    str(i): t
    for i, (q, t) in enumerate(eval_pairs)
}
queries = {
    str(i): q
    for i, (q, t) in enumerate(eval_pairs)
}
# Mapowanie pytanie -> właściwy fragment (1-do-1)
relevant_docs = {
    str(i): {str(i)}
    for i in range(len(eval_pairs))
}

evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name='przemkow-eval',
    mrr_at_k=[10],
    ndcg_at_k=[10],
    accuracy_at_k=[1, 5],
    precision_recall_at_k=[5],
    map_at_k=[10],
)

# Sprawdź wynik PRZED treningiem
print('=== Wynik PRZED treningiem ===')
results_before = evaluator(model)
print(results_before)

## 7. Konfiguracja treningu

Parametry dobrane pod T4 (15 GB VRAM). Dla A100 możesz zwiększyć `per_device_train_batch_size` do 128.

In [ ]:
OUTPUT_DIR = '/content/drive/MyDrive/historyk_ai/st-przemkow-mpnet'

# ── Hiperparametry ────────────────────────────────────────────
NUM_EPOCHS          = 3        # 3 dało +0.5pp vs 5 epok w poprzednim treningu
BATCH_SIZE          = 64       # T4: 64 bezpiecznie; A100: spróbuj 128
LEARNING_RATE       = 2e-5
WARMUP_RATIO        = 0.1      # 10% kroków = warmup
EVAL_STEPS          = 200      # co ile kroków ewaluacja
SAVE_STEPS          = 200
# ─────────────────────────────────────────────────────────────

training_args = SentenceTransformerTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    bf16=False,         # T4 nie obsługuje bf16; zmień na True dla A100
    fp16=True,          # T4 obsługuje fp16
    evaluation_strategy='steps',
    eval_steps=EVAL_STEPS,
    save_strategy='steps',
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_przemkow-eval_cosine_accuracy@1',
    greater_is_better=True,
    logging_steps=50,
    report_to='none',   # zmień na 'wandb' jeśli chcesz śledzić w W&B
    dataloader_drop_last=True,  # ważne dla MNR Loss (pełne batche)
)

loss_fn = MultipleNegativesRankingLoss(model=model)

print('Konfiguracja gotowa.')
print(f'Kroki na epokę: {len(train_pairs) // BATCH_SIZE}')
print(f'Łączna liczba kroków: {NUM_EPOCHS * len(train_pairs) // BATCH_SIZE}')

## 8. Trening

In [ ]:
trainer = SentenceTransformerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    loss=loss_fn,
    evaluator=evaluator,
)

trainer.train()

## 9. Ewaluacja końcowa

In [ ]:
print('=== Wynik PO treningu (najlepszy checkpoint) ===')
results_after = evaluator(model)

# Porównanie kluczowych metryk
key_metrics = [
    'przemkow-eval_cosine_accuracy@1',
    'przemkow-eval_cosine_accuracy@5',
    'przemkow-eval_cosine_mrr@10',
    'przemkow-eval_cosine_ndcg@10',
]

print(f'\n{"Metryka":<50} {"Przed":>8} {"Po":>8} {"Delta":>8}')
print('-' * 76)
for m in key_metrics:
    before = results_before.get(m, 0)
    after  = results_after.get(m, 0)
    delta  = after - before
    sign   = '+' if delta >= 0 else ''
    print(f'{m:<50} {before:>8.4f} {after:>8.4f} {sign}{delta:>7.4f}')

## 10. Zapis modelu

In [ ]:
FINAL_MODEL_PATH = '/content/drive/MyDrive/historyk_ai/st-przemkow-mpnet-final'

model.save(FINAL_MODEL_PATH)
print(f'Model zapisany w: {FINAL_MODEL_PATH}')

# Sprawdź rozmiar
import subprocess
result = subprocess.run(['du', '-sh', FINAL_MODEL_PATH], capture_output=True, text=True)
print('Rozmiar na dysku:', result.stdout.strip())

## 11. Test manualny po treningu

In [ ]:
import torch
from sentence_transformers import util

# Wczytaj zapisany model (możesz też użyć `model` bezpośrednio)
# model_test = SentenceTransformer(FINAL_MODEL_PATH)
model_test = model

test_queries = [
    'Kiedy urodził się Ernest Günther książę Szlezwika-Holsztynu?',
    'Co znajdowało się w parku w Przemkowie?',
    'Jakie zawody rzemieślnicze uprawiano w Przemkowie w XIX wieku?',
]

test_corpus = [
    'Ernest Günther urodził się 11 sierpnia 1863 roku.',
    'W parku w Przemkowie znajdowało się kąpielisko z wyspą.',
    'W Przemkowie działali kowale, krawcy i szewcy.',
    'Książę przybył pociągiem z Berlina do Wrocławia.',  # distractor
    'Temperatura wody w kąpielisku wynosiła 22 stopnie.',  # distractor
]

q_emb = model_test.encode(test_queries, convert_to_tensor=True, normalize_embeddings=True)
c_emb = model_test.encode(test_corpus,  convert_to_tensor=True, normalize_embeddings=True)

scores = util.cos_sim(q_emb, c_emb)

print('Wyniki podobieństwa kosinus:\n')
for i, query in enumerate(test_queries):
    print(f'Pytanie: {query}')
    top = torch.topk(scores[i], k=2)
    for score, idx in zip(top.values, top.indices):
        print(f'  [{score:.4f}] {test_corpus[idx]}')
    print()

## 12. Wgranie modelu na serwer (opcjonalne)

Skopiuj model z Dysku Google do kontenera 101 na GMKtecu.

```bash
# Na Windowsie (PowerShell) — pobierz z Dysku i wyślij przez SCP
# Najpierw pobierz folder st-przemkow-mpnet-final z Dysku Google na lokalny dysk

# Następnie prześlij na serwer:
scp -r C:\Users\TwojaNazwa\Downloads\st-przemkow-mpnet-final `
    eugeniusz@192.168.1.24:/tmp/

# Na Proxmoxie:
pct exec 101 -- mkdir -p /home/eugeniusz/models
pct exec 101 -- bash -c 'cp -r /tmp/st-przemkow-mpnet-final /home/eugeniusz/models/'

# W kodzie historyk_ai — zmień nazwę modelu:
# EMBEDDING_MODEL = '/home/eugeniusz/models/st-przemkow-mpnet-final'
```

Pamiętaj: ten model **nie używa prefixów** `query:` / `passage:` — w odróżnieniu od `multilingual-e5-base`.  
Usuń prefixy z kodu indeksowania i wyszukiwania jeśli przechodzisz z e5-przemkow na ten model.

---
## Notatki o hiperparametrach

| Parametr | Wartość | Uwaga |
|---|---|---|
| `NUM_EPOCHS` | 3 | Optymalny z poprzedniego treningu; 5 epok dało gorsze wyniki |
| `BATCH_SIZE` | 64 | Większy batch = lepsze hard negatives dla MNR Loss |
| `LEARNING_RATE` | 2e-5 | Standardowy dla fine-tuningu modeli ~110M |
| `WARMUP_RATIO` | 0.1 | 10% kroków na warmup learning rate |
| Loss | MNR | Nie wymaga jawnych negatywów; inne przykłady w batchu = negatives |
| `fp16` | True | T4; zmień na `bf16=True` dla A100 |
| `dataloader_drop_last` | True | Konieczne dla MNR Loss — ostatni niepełny batch psuje trening |

### Eksperymenty warte wypróbowania
- Zwiększ `BATCH_SIZE` do 128 na A100 — MNR korzysta na większych batchach
- Wypróbuj `CachedMultipleNegativesRankingLoss` (obsługuje batch_size=512+)
- Dodaj `GISTEmbedLoss` jako alternatywę dla lepszego doboru negatywów

## 13. Pobranie modelu jako ZIP do folderu Downloads

In [ ]:
import os
import zipfile
from google.colab import files

ZIP_PATH = '/content/st-przemkow-mpnet-final.zip'

print('Pakuję model do ZIP...')
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, filenames in os.walk(FINAL_MODEL_PATH):
        for filename in filenames:
            filepath = os.path.join(root, filename)
            arcname = os.path.relpath(filepath, os.path.dirname(FINAL_MODEL_PATH))
            zf.write(filepath, arcname)

size_mb = os.path.getsize(ZIP_PATH) / 1024 / 1024
print(f'ZIP gotowy: {ZIP_PATH} ({size_mb:.1f} MB)')
print('Rozpoczynam pobieranie...')

files.download(ZIP_PATH)